# Test nhanh Logic Inference: Answer + Explanation

Dung base **Qwen2.5-3B-Instruct** (chua fine-tune) de test:
- Dua premises (NL + FOL) + question vao
- Model sinh ra Answer + Explanation

Chay tren Vast.ai / Colab co GPU.

In [1]:
# Cell 1: Load model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded on {model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded on cuda:0


In [2]:
# Cell 2: Prompts (giong prompts.py)

SYSTEM_PROMPT = (
    "You are an expert in formal logic and educational reasoning. "
    "Given premises in natural language (and optionally first-order logic), "
    "answer the question with the correct label and a clear explanation "
    "that cites relevant premises when appropriate."
)

def build_user_prompt(premises_nl, premises_fol, question):
    """Ghep premises + question thanh user message."""
    blocks = []

    # FOL block
    if premises_fol:
        fol_lines = "\n".join(f"{i}. {f}" for i, f in enumerate(premises_fol, 1))
        blocks.append(f"Premises (FOL):\n{fol_lines}")

    # NL block
    nl_lines = "\n".join(f"{i}. {p}" for i, p in enumerate(premises_nl, 1))
    blocks.append(f"Premises (NL):\n{nl_lines}")

    premises_block = "\n\n".join(blocks)
    return f"{premises_block}\n\nQuestion:\n{question}"

print("Prompts ready")

Prompts ready


In [ ]:
# Cell 3: Ham generate

def logic_inference(premises_nl, premises_fol, question, max_new_tokens=512):
    """Sinh Answer + Explanation tu premises + question."""
    user_content = build_user_prompt(premises_nl, premises_fol, question)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    input_len = enc["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    result = tokenizer.decode(out[0, input_len:], skip_special_tokens=True)
    return result

print("logic_inference() ready")

logic_inference() ready


In [4]:
# Cell 4: TEST 1 - MCQ (4 dap an)

premises_nl_1 = [
    "If a curriculum is well-structured and has exercises, it enhances student engagement.",
    "If a curriculum enhances student engagement and provides access to advanced resources, it enhances critical thinking.",
    "The curriculum is well-structured.",
    "Faculty prioritize pedagogical training and curriculum development.",
    "The curriculum has exercises.",
    "The curriculum provides access to advanced resources.",
]

premises_fol_1 = [
    "\u2200x (WellStructured(x) \u2227 HasExercises(x) \u2192 EnhancesStudentEngagement(x))",
    "\u2200x (EnhancesStudentEngagement(x) \u2227 ProvidesAdvancedResources(x) \u2192 EnhancesCriticalThinking(x))",
    "WellStructured(curriculum)",
    "PrioritizePedagogicalTraining(faculty) \u2227 PrioritizeCurriculumDevelopment(faculty)",
    "HasExercises(curriculum)",
    "ProvidesAdvancedResources(curriculum)",
]

question_1 = """Based on the premises, what can we conclude about the curriculum?
A. It enhances student engagement but not critical thinking
B. It enhances critical thinking
C. It needs more resources to enhance critical thinking
D. Faculty training is insufficient"""

print("="*60)
print("TEST 1: MCQ (expected: B)")
print("="*60)

# Chi NL
print("\n--- Chi NL ---")
r1_nl = logic_inference(premises_nl_1, [], question_1)
print(r1_nl)

# NL + FOL
print("\n--- NL + FOL ---")
r1_both = logic_inference(premises_nl_1, premises_fol_1, question_1)
print(r1_both)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TEST 1: MCQ (expected: B)

--- Chi NL ---
To determine the conclusion based on the given premises, let's analyze each premise step by step:

1. **If a curriculum is well-structured and has exercises, it enhances student engagement.**
   - Premise 3 states: "The curriculum is well-structured."
   - Premise 5 states: "The curriculum has exercises."
   - Therefore, from premises 3 and 5, we can conclude that the curriculum enhances student engagement (by applying the rule from premise 1).

2. **If a curriculum enhances student engagement and provides access to advanced resources, it enhances critical thinking.**
   - From premise 1, we know the curriculum enhances student engagement.
   - Premise 6 states: "The curriculum provides access to advanced resources."
   - Therefore, by applying the rule from premise 2, we can conclude that the curriculum enhances critical thinking.

Given these conclusions, let's evaluate the provided options:

A. It enhances student engagement but not critical

In [ ]:
# Cell 5: TEST 2 - Yes/No/Unknown

premises_nl_2 = [
    "All students who complete the internship receive a certificate.",
    "John is a student.",
    "John has completed the internship.",
    "Only students with certificates are eligible for the job fair.",
]

premises_fol_2 = [
    "\u2200x (Student(x) \u2227 CompletedInternship(x) \u2192 ReceivesCertificate(x))",
    "Student(John)",
    "CompletedInternship(John)",
    "\u2200x (EligibleForJobFair(x) \u2192 ReceivesCertificate(x))",
]

question_2 = "Is John eligible for the job fair?"

print("="*60)
print("TEST 2: Yes/No/Unknown (expected: Yes)")
print("="*60)

print("\n--- Chi NL ---")
r2_nl = logic_inference(premises_nl_2, [], question_2)
print(r2_nl)

print("\n--- NL + FOL ---")
r2_both = logic_inference(premises_nl_2, premises_fol_2, question_2)
print(r2_both)

TEST 2: Yes/No/Unknown (expected: Yes)

--- Chi NL ---
To determine if John is eligible for the job fair, we need to analyze the given premises step by step:

1. **All students who complete the internship receive a certificate.**
   - This premise establishes a relationship between completing an internship and receiving a certificate. If a student completes an internship, they will receive a certificate.

2. **John is a student.**
   - This premise tells us that John belongs to the category of students.

3. **John has completed the internship.**
   - This premise indicates that John has successfully completed the internship.

4. **Only students with certificates are eligible for the job fair.**
   - This premise specifies that eligibility for the job fair is contingent upon having a certificate. 

Now, let's connect these premises logically:

- From Premise 1, since John has completed the internship, he should receive a certificate (Premise 1 applies to all students who complete the in

In [ ]:
# Cell 6: TEST 3 - Unknown case

premises_nl_3 = [
    "All birds can fly.",
    "Penguins are birds.",
    "Tweety is a penguin.",
]

premises_fol_3 = [
    "\u2200x (Bird(x) \u2192 CanFly(x))",
    "\u2200x (Penguin(x) \u2192 Bird(x))",
    "Penguin(Tweety)",
]

question_3 = "Can Tweety swim?"

print("="*60)
print("TEST 3: Unknown (khong co thong tin ve swim)")
print("="*60)

print("\n--- NL + FOL ---")
r3 = logic_inference(premises_nl_3, premises_fol_3, question_3)
print(r3)

In [ ]:
# Cell 7: Load 1 sample that tu dataset that
import pandas as pd
import ast

# Doc CSV
df = pd.read_csv("../data/processed/test.csv")

# Lay 1 row
row = df.iloc[0]
real_nl = ast.literal_eval(row["premises_nl"])
real_fol = ast.literal_eval(row["premises_fol"]) if pd.notna(row["premises_fol"]) else []
real_question = row["question"]
real_answer = row["answer"]
real_explanation = row["explanation"]

print("GROUND TRUTH:")
print(f"  Answer: {real_answer}")
print(f"  Explanation: {real_explanation[:200]}...")
print()

# Inference
print("MODEL OUTPUT:")
r_real = logic_inference(real_nl, real_fol, real_question)
print(r_real)

In [ ]:
# Cell 8: So sanh nhieu sample
import re

def extract_answer(text):
    """Trich answer tu output."""
    m = re.search(r"(?im)^\s*answer\s*:\s*(.+?)(?:\n|$)", text)
    if m:
        ans = m.group(1).strip().split()[0].rstrip(".,;:")
        return ans
    return text.strip().split()[0] if text.strip() else "?"

N_SAMPLES = 5  # so sample muon test
correct = 0

for idx in range(min(N_SAMPLES, len(df))):
    row = df.iloc[idx]
    nl = ast.literal_eval(row["premises_nl"])
    fol = ast.literal_eval(row["premises_fol"]) if pd.notna(row["premises_fol"]) else []
    q = row["question"]
    gold = row["answer"]

    output = logic_inference(nl, fol, q)
    pred = extract_answer(output)
    match = "OK" if pred == gold else "MISS"
    if pred == gold:
        correct += 1

    print(f"[{idx}] Gold={gold} | Pred={pred} | {match}")

print(f"\nAccuracy: {correct}/{min(N_SAMPLES, len(df))}")